# Transformer Block 완전 구현 - 실습 코드 1: 완전한 Transformer Encoder

- Tutorial ID: `expand-transformer-block-impl`
- Tutorial: Transformer Block 완전 구현
- Section ID: `expand-transformer-block-impl-code-1`
- Section: 실습 코드 1: 완전한 Transformer Encoder


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: 완전한 Transformer Encoder
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeedForward(nn.Module):
        def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        def forward(self, x):
        return self.net(x)

class EncoderBlock(nn.Module):
        def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
                        d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
                self.dropout = nn.Dropout(dropout)

        def forward(self, x, src_mask=None):
        # Pre-LN Self-Attention
        normed = self.norm1(x)
                attn_out, _ = self.self_attn(normed, normed, normed, attn_mask=src_mask)
                x = x + self.dropout(attn_out)
        # Pre-LN FFN
                x = x + self.dropout(self.ffn(self.norm2(x)))
        return x

class TransformerEncoder(nn.Module):
        def __init__(self, vocab_size, d_model=512, num_heads=8,
                 d_ff=2048, num_layers=6, max_seq_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Embedding(max_seq_len, d_model)
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff)
                        for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

        def forward(self, x):
                seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device)
                x = self.embedding(x) + self.pos_encoding(positions)
                for layer in self.layers:
                        x = layer(x)
        return self.norm(x)

# 테스트
model = TransformerEncoder(vocab_size=30000, d_model=256, num_heads=4, d_ff=1024, num_layers=4)
x = torch.randint(0, 30000, (2, 64))
out = model(x)
print(f"Input: {x.shape} → Output: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")